In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing     import LabelEncoder, StandardScaler
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif
from sklearn.ensemble          import RandomForestClassifier
from sklearn.model_selection   import StratifiedKFold
from sklearn.metrics           import (accuracy_score, f1_score, precision_score,
                                       recall_score, roc_auc_score, classification_report,
                                       confusion_matrix, ConfusionMatrixDisplay, roc_curve)


DATA_PATH = r"C:\Users\Admin\OneDrive\Documents\data to work on.csv"

df    = pd.read_csv(DATA_PATH)
X     = df.drop(columns=["Class"])
y     = df["Class"]
le    = LabelEncoder()
y_enc = le.fit_transform(y)

print(f"Dataset shape : {df.shape}")
print(f"Class counts  :\n{y.value_counts()}\n")

# ── 2. PREPROCESSING 
vt = VarianceThreshold(threshold=0.01)
X  = pd.DataFrame(vt.fit_transform(X), columns=X.columns[vt.get_support()])

corr  = X.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
X     = X.drop(columns=[c for c in upper.columns if any(upper[c] > 0.90)])

scaler = StandardScaler()
Xs     = scaler.fit_transform(X)
print(f"After preprocessing : {X.shape[1]} features")

# ── 3. FEATURE SELECTION
mi    = mutual_info_classif(Xs, y_enc, random_state=42)
top80 = pd.Series(mi, index=X.columns).nlargest(80).index.tolist()
idx80 = [X.columns.get_loc(f) for f in top80]

rf_fs = RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                random_state=42, n_jobs=-1)
rf_fs.fit(Xs[:, idx80], y_enc)
fi    = pd.Series(rf_fs.feature_importances_, index=top80).sort_values(ascending=False)
top40 = fi.head(40).index.tolist()
Xf    = Xs[:, [X.columns.get_loc(f) for f in top40]]
print(f"After feature selection : {Xf.shape[1]} features")
print(f"Top 5: {top40[:5]}\n")

# ── 4. MANUAL SMOTE 
def manual_smote(X, y, minority_class=1, k=3, random_state=42):
    rng      = np.random.RandomState(random_state)
    X_min    = X[y == minority_class]
    n_needed = int((y == 0).sum() - (y == 1).sum())
    synthetic = []
    for _ in range(n_needed):
        idx        = rng.randint(0, len(X_min))
        dists      = np.linalg.norm(X_min - X_min[idx], axis=1)
        dists[idx] = np.inf
        neighbors  = np.argsort(dists)[:k]
        nn_idx     = rng.choice(neighbors)
        alpha      = rng.random()
        synthetic.append(X_min[idx] + alpha * (X_min[nn_idx] - X_min[idx]))
    return (np.vstack([X, np.vstack(synthetic)]),
            np.hstack([y, np.ones(len(synthetic), dtype=int)]))

# ── 5. MODEL 
model = RandomForestClassifier(
    n_estimators     = 600,
    max_depth        = 10,
    min_samples_leaf = 2,
    max_features     = 'sqrt',
    class_weight     = 'balanced',
    random_state     = 42,
    n_jobs           = -1
)

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
acc_l, f1_l, prec_l, rec_l, auc_l = [], [], [], [], []
all_yt, all_yp, all_yprob = [], [], []

print("── Training (10-Fold CV + SMOTE per fold) ──")
for fold, (tr_idx, te_idx) in enumerate(cv.split(Xf, y_enc), 1):
    X_tr, X_te = Xf[tr_idx], Xf[te_idx]
    y_tr, y_te = y_enc[tr_idx], y_enc[te_idx]
    X_tr_s, y_tr_s = manual_smote(X_tr, y_tr, k=3)
    model.fit(X_tr_s, y_tr_s)
    yp    = model.predict(X_te)
    yprob = model.predict_proba(X_te)[:, 1]
    acc_l.append(accuracy_score(y_te, yp))
    f1_l.append(f1_score(y_te, yp, zero_division=0))
    prec_l.append(precision_score(y_te, yp, zero_division=0))
    rec_l.append(recall_score(y_te, yp, zero_division=0))
    try:    auc_l.append(roc_auc_score(y_te, yprob))
    except: auc_l.append(np.nan)
    all_yt.extend(y_te); all_yp.extend(yp); all_yprob.extend(yprob)
    print(f"  Fold {fold:2d} | Acc={acc_l[-1]:.3f}  F1={f1_l[-1]:.3f}  "
          f"Recall={rec_l[-1]:.3f}  AUC={auc_l[-1]:.3f}")

# ── 6. RESULTS 
all_yt = np.array(all_yt); all_yp = np.array(all_yp); all_yprob = np.array(all_yprob)

print("\n── Final Performance ──")
print(f"  Accuracy  : {np.mean(acc_l):.4f}  ± {np.std(acc_l):.4f}")
print(f"  F1        : {np.mean(f1_l):.4f}  ± {np.std(f1_l):.4f}")
print(f"  Precision : {np.mean(prec_l):.4f}  ± {np.std(prec_l):.4f}")
print(f"  Recall    : {np.mean(rec_l):.4f}  ± {np.std(rec_l):.4f}")
print(f"  ROC-AUC   : {np.nanmean(auc_l):.4f}  ± {np.nanstd(auc_l):.4f}")
print("\n── Classification Report ──")
print(classification_report(all_yt, all_yp, target_names=le.classes_))

# ── 7. PLOTS 
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Random Forest + SMOTE — 10-Fold CV", fontsize=13, fontweight='bold')

ConfusionMatrixDisplay(confusion_matrix(all_yt, all_yp),
    display_labels=le.classes_).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title("Confusion Matrix")

fpr, tpr, _ = roc_curve(all_yt, all_yprob)
auc_val      = roc_auc_score(all_yt, all_yprob)
axes[1].plot(fpr, tpr, color='#e74c3c', lw=2, label=f'AUC = {auc_val:.3f}')
axes[1].plot([0,1],[0,1],'k--',lw=1); axes[1].fill_between(fpr,tpr,alpha=0.1,color='#e74c3c')
axes[1].set_xlabel("FPR"); axes[1].set_ylabel("TPR"); axes[1].set_title("ROC Curve")
axes[1].legend(); axes[1].grid(alpha=0.3)

fold_x = np.arange(1, 11)
axes[2].plot(fold_x, acc_l,'o-',color='#2ecc71',lw=2,ms=7,label='Accuracy')
axes[2].plot(fold_x, f1_l, 's-',color='#3498db',lw=2,ms=7,label='F1 (Toxic)')
axes[2].plot(fold_x, rec_l,'^-',color='#e74c3c',lw=2,ms=7,label='Recall (Toxic)')
for v, c in zip([np.mean(acc_l),np.mean(f1_l),np.mean(rec_l)],['#2ecc71','#3498db','#e74c3c']):
    axes[2].axhline(v, color=c, ls=':', lw=1.5)
axes[2].set_xlabel("Fold"); axes[2].set_ylabel("Score"); axes[2].set_title("Per-Fold Metrics")
axes[2].legend(); axes[2].set_xticks(fold_x); axes[2].set_ylim(0,1.05); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("performance_final.png", dpi=130, bbox_inches='tight')
plt.close()
print("\n[Saved] performance_final.png")

# ── 8. FEATURE IMPORTANCES 
model.fit(*manual_smote(Xf, y_enc, k=3))
fi_final = pd.Series(model.feature_importances_, index=top40).sort_values(ascending=False).head(15)
fig2, ax2 = plt.subplots(figsize=(9, 5))
fi_final[::-1].plot(kind='barh', ax=ax2, color='#3498db', edgecolor='white')
ax2.set_title("Top 15 Feature Importances", fontweight='bold')
ax2.set_xlabel("Importance"); ax2.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig("feature_importances_final.png", dpi=130, bbox_inches='tight')
plt.close()
print("[Saved] feature_importances_final.png")
print("\nDone ✓")

Dataset shape : (171, 1204)
Class counts  :
Class
NonToxic    115
Toxic        56
Name: count, dtype: int64

After preprocessing : 420 features
After feature selection : 40 features
Top 5: ['MDEC-23', 'ATSC8i', 'SHaaCH', 'CIC1', 'AATSC8i']

── Training (10-Fold CV + SMOTE per fold) ──
  Fold  1 | Acc=0.611  F1=0.364  Recall=0.333  AUC=0.486
  Fold  2 | Acc=0.588  F1=0.222  Recall=0.200  AUC=0.500
  Fold  3 | Acc=0.706  F1=0.545  Recall=0.600  AUC=0.650
  Fold  4 | Acc=0.765  F1=0.667  Recall=0.800  AUC=0.783
  Fold  5 | Acc=0.765  F1=0.500  Recall=0.400  AUC=0.767
  Fold  6 | Acc=0.765  F1=0.714  Recall=0.833  AUC=0.803
  Fold  7 | Acc=0.706  F1=0.545  Recall=0.500  AUC=0.682
  Fold  8 | Acc=0.529  F1=0.333  Recall=0.333  AUC=0.606
  Fold  9 | Acc=0.765  F1=0.667  Recall=0.667  AUC=0.833
  Fold 10 | Acc=0.647  F1=0.400  Recall=0.333  AUC=0.652

── Final Performance ──
  Accuracy  : 0.6846  ± 0.0817
  F1        : 0.4958  ± 0.1546
  Precision : 0.5113  ± 0.1361
  Recall    : 0.5000  ± 0.